# Exp 4 — Breadth: newer detectors, instance masks, and a setting where batching pays off

Exps 1–3 showed the problem on six detectors and a training-free fix (`self`) that exactly restores the batch-1 output. Reviewers will ask three things; this notebook answers them in three parts.

**Part A — "Is this only old models?"** Six more models, measured with normal batching (batch 1 vs random / sorted / max-padding batches of 8):
- DETR descendants: **Conditional DETR**, **DAB-DETR**;
- a plain ViT detector: **YOLOS-S** (a different leak route: position embeddings resized to the canvas);
- the most widely used open-vocabulary detector: **Grounding DINO-T**, prompted with all 80 COCO class names;
- a transformer instance segmenter: **Mask2Former (Swin-S)**, with boxes taken from its masks;
- a modern no-padding control: **RT-DETRv2**.

The `self` fix is also applied to **Conditional DETR**, whose backbone has the same structure as DETR's, to test whether the fix transfers without new code.

**Part B — "Do masks change too?"** Mask R-CNN, comparing every instance mask (mask IoU) and mask AP, normal vs `self`.

**Part C — "Why not just use batch 1?"** In Exp 3, full-size fp32 images saturated the T4, so batching gave no speed-up. Here Faster R-CNN and RetinaNet run in **half precision at a smaller input size (short side 512)**, where batching does pay. Images are converted to GPU tensors beforehand, so timing measures the model only. We also compare against the obvious alternative fix, **exact-shape bucketing**: only batch images whose own canvases are identical. That is exact without touching the model, but the batches are less full.

**Claims (fixed before running):**
- **G1 (breadth):** at least 3 of the 5 new padding-based models change ≥ 10% of images under random batch-8; the RT-DETRv2 control changes < 0.5%.
- **G2 (fix transfers):** `self` on Conditional DETR changes ≤ 0.5% of images vs normal batch 1, with |ΔAP| ≤ 0.05.
- **G3 (masks):** normal random batching changes ≥ 10% of Mask R-CNN images' masks; `self` changes ≤ 0.5%, with |Δ mask AP| ≤ 0.05.
- **G4 (speed where batching pays):** (a) normal sorted batch-8 is ≥ 1.3× batch-1 throughput; (b) `self` sorted keeps ≥ 0.9× of normal sorted; (c) `self` stays exact in half precision (≤ 0.5% of images changed vs half-precision batch 1).

**Kaggle:** GPU T4 x2 or P100, Internet ON, nothing to attach. About 1.5–2 hours; resumes per model/part if restarted. A model that fails to load is skipped and reported.

In [ ]:
!pip install -q "transformers==5.17.0" pycocotools timm

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")   # less memory fragmentation
import io, sys, glob, json, time, math, random, zipfile, platform, datetime, contextlib, urllib.request, dataclasses, gc
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torchvision
import torch.nn.functional as F
import torch.nn as nn
import inspect
from pycocotools import mask as mask_utils
import matplotlib.pyplot as plt
from PIL import Image
import transformers
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

OUT = "/kaggle/working/exp4"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

ENV = {
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
if ENV["gpu"] is None:
    print("\n⚠⚠ NO GPU. Stop, set Accelerator to GPU in the right panel, and Run All again. ⚠⚠")

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

## 1. Configuration

In [ ]:
CFG = {
    "n_images": 500,
    "seed": 0,
    "batch_size": 8,
    "models_A": ["conditional_detr", "dab_detr", "yolos_small", "grounding_dino_tiny", "mask2former_swin_s", "rtdetr_v2_r50"],
    "control_models": ["rtdetr_v2_r50"],
    "flip_thr_per_model": {"grounding_dino_tiny": 0.3},   # its confident scores are lower than closed-set detectors'
    "model_B": "maskrcnn_v2",
    "models_C": ["frcnn_v2", "retinanet_v2"],
    "speed_min_size": 512, "speed_max_size": 853,
    "micro_batch": {"grounding_dino_tiny": 2, "mask2former_swin_s": 4, "yolos_small": 4},
    "flip_score_thr": 0.5,
    "flip_iou": 0.9,
    "coco_root": "/kaggle/input",
    "coco_download_dir": "/kaggle/working/coco",
    "claims": {"G1_flip": 0.10, "G1_models": 3, "G1_ctrl": 0.005, "G2_flip": 0.005, "G2_dap": 0.05,
               "G3_normal": 0.10, "G3_self": 0.005, "G3_dap": 0.05, "G4_batching": 1.3, "G4_self": 0.9, "G4_exact": 0.005},
}
AMP = False          # half precision; switched on only in Part C
RUN_PARTS = {"A": True, "B": True, "C": True}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

## 2. COCO val2017 (same 500 images as before)

In [ ]:
def find_attached_coco(root):
    anns = glob.glob(os.path.join(root, "**", "instances_val2017.json"), recursive=True)
    probe = glob.glob(os.path.join(root, "**", "000000000139.jpg"), recursive=True)
    if anns and probe:
        return anns[0], os.path.dirname(probe[0])
    return None, None

ANN, IMG_DIR = find_attached_coco(CFG["coco_root"])
if ANN is None:
    os.makedirs(CFG["coco_download_dir"], exist_ok=True)
    ANN = os.path.join(CFG["coco_download_dir"], "instances_val2017.json")
    IMG_DIR = os.path.join(CFG["coco_download_dir"], "val2017")
    if not os.path.exists(ANN):
        zp = os.path.join(CFG["coco_download_dir"], "ann.zip")
        print("downloading COCO annotations (~250 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
        with zipfile.ZipFile(zp) as z, open(ANN, "wb") as f:
            f.write(z.read("annotations/instances_val2017.json"))
        os.remove(zp)

with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(ANN)
NAME2CAT = {c["name"].strip().lower(): c["id"] for c in GT.loadCats(GT.getCatIds())}
IDS = sorted(random.Random(CFG["seed"]).sample(sorted(GT.getImgIds()), CFG["n_images"]))
INFO = GT.loadImgs(IDS)
os.makedirs(IMG_DIR, exist_ok=True)
def fetch(info):
    p = os.path.join(IMG_DIR, info["file_name"])
    if not os.path.exists(p):
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{info['file_name']}", p)
    return p
with ThreadPoolExecutor(16) as ex:
    paths = list(ex.map(fetch, INFO))
IMGS = [Image.open(p).convert("RGB") for p in paths]
N = len(IMGS)
print(f"{N} images; first ids {IDS[:3]} (Day 0 used seed 0 too, so these are the same images)")

## 3. The `self` fix (unchanged from Exp 3, except GroupNorm statistics are computed in fp32 under half precision)

In [ ]:
class Fix:
    COMPONENTS = ("conv", "norm", "own", "exactmask")
    def __init__(self):
        for c in self.COMPONENTS:
            setattr(self, c, False)
        self.valid, self.canvas, self.trivial, self._cache = None, None, True, {}

    def set(self, components):
        for c in self.COMPONENTS:
            setattr(self, c, c in components)

    def any_on(self):
        return any(getattr(self, c) for c in self.COMPONENTS)

    def begin(self, valid, canvas):
        self.valid = [tuple(int(v) for v in s) for s in valid]
        self.canvas = tuple(int(c) for c in canvas)
        self.trivial = all(v == self.canvas for v in self.valid)

    def end(self):
        self.valid = None

    def active(self):
        return self.valid is not None and not self.trivial

    def applies(self, x):
        return self.active() and x.dim() == 4 and x.shape[0] == len(self.valid)

    @staticmethod
    def _stride(n, N):
        for s in (1, 2, 4, 8, 16, 32, 64, 128, 256):
            if math.ceil(N / s) == n:
                return s
        return N / n

    def extents(self, H, W):
        """each image's valid (rows, cols) at a feature map of size H x W"""
        sh, sw = self._stride(H, self.canvas[0]), self._stride(W, self.canvas[1])
        return [(math.ceil(h / sh), math.ceil(w / sw)) for h, w in self.valid]

    def mask(self, x):
        H, W = x.shape[-2:]
        key = (H, W, tuple(self.valid), self.canvas, x.device, x.dtype)
        if key not in self._cache:
            m = torch.zeros(len(self.valid), 1, H, W, device=x.device, dtype=x.dtype)
            for i, (h, w) in enumerate(self.extents(H, W)):
                m[i, :, :h, :w] = 1
            if len(self._cache) > 64:
                self._cache.clear()
            self._cache[key] = m
        return self._cache[key]

def _mask_conv_inputs(root, fix):
    """Zero the padding at the input of every conv / max-pool: each layer then sees exactly the zeros its
    batch-1 run sees past the canvas edge (activations are >= 0 after ReLU, so max-pool is unaffected)."""
    def pre(mod, inp):
        x = inp[0]
        if fix.conv and fix.applies(x):
            return (x * fix.mask(x),) + tuple(inp[1:])
    for m in root.modules():
        if isinstance(m, (nn.Conv2d, nn.MaxPool2d)):
            m.register_forward_pre_hook(pre)

def _masked_groupnorm(root, fix):
    """GroupNorm statistics over each image's own valid area only."""
    n = 0
    for mod in root.modules():
        if isinstance(mod, nn.GroupNorm):
            orig = mod.forward
            def fwd(x, mod=mod, orig=orig):
                if not (fix.norm and fix.applies(x)):
                    return orig(x)
                m = fix.mask(x).float(); B, C, H, W = x.shape; G = mod.num_groups
                xg, mg = x.float().reshape(B, G, C // G, H, W), m.reshape(B, 1, 1, H, W)
                cnt = mg.sum(dim=(2, 3, 4), keepdim=True) * (C // G)
                mean = (xg * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                var = (((xg - mean) ** 2) * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                y = ((xg - mean) / torch.sqrt(var + mod.eps)).reshape(B, C, H, W)
                if mod.affine:
                    y = y * mod.weight.float()[None, :, None, None] + mod.bias.float()[None, :, None, None]
                return y.to(x.dtype)
            mod.forward = fwd; n += 1
    return n

def _suppress_padding_scores(conv, fix):
    """Classification logits at positions that do not exist in the image's batch-1 run -> -1e4."""
    def hook(mod, inp, out):
        if fix.conv and fix.applies(out):
            return out.masked_fill(fix.mask(out) == 0, -1e4)
    conv.register_forward_hook(hook)

def _own_anchors(ag, fix):
    """Each image gets the anchors its own batch-1 canvas would give: torchvision spaces anchors at
    canvas // grid, so we use the image's own canvas and grid. Anchors at positions that do not exist in
    its batch-1 run become zero-size boxes, which torchvision removes."""
    orig = ag.forward
    def fwd(image_list, feature_maps):
        if not (fix.own and fix.active()):
            return orig(image_list, feature_maps)
        grid_sizes = [list(f.shape[-2:]) for f in feature_maps]
        Hc, Wc = image_list.tensors.shape[-2:]
        dtype, device = feature_maps[0].dtype, feature_maps[0].device
        ag.set_cell_anchors(dtype, device)
        n_cell = [ca.shape[0] for ca in ag.cell_anchors]
        out = []
        for ch, cw in fix.valid:
            strides, keep = [], []
            for (gh, gw), a in zip(grid_sizes, n_cell):
                sh, sw = 2 ** round(math.log2(Hc / gh)), 2 ** round(math.log2(Wc / gw))
                oh, ow = math.ceil(ch / sh), math.ceil(cw / sw)
                strides.append([torch.tensor(ch // oh, dtype=torch.int64, device=device),
                                torch.tensor(cw // ow, dtype=torch.int64, device=device)])
                k = torch.zeros(gh, gw, dtype=dtype, device=device); k[:oh, :ow] = 1
                keep.append(k.reshape(-1).repeat_interleave(a))
            anc = ag.grid_anchors(grid_sizes, strides)
            out.append(torch.cat([x * k[:, None] for x, k in zip(anc, keep)]))
        return out
    ag.forward = fwd

def install_fix_tv(model, fix):
    heads = [model.rpn.head] if hasattr(model, "rpn") else [model.head]
    for root in [model.backbone] + heads:
        _mask_conv_inputs(root, fix)
    has_roi = hasattr(model, "roi_heads")
    def zero_outputs(mod, inp, out):
        if not (fix.conv and fix.active()):
            return
        new = type(out)()
        for k, v in out.items():
            v = v * fix.mask(v)
            if fix.own and has_roi:
                # RoIAlign at batch 1 clamps samples at the canvas edge to the last row/column; copying that
                # row/column one step outward reproduces this inside a bigger canvas.
                H, W = v.shape[-2:]
                for i, (hl, wl) in enumerate(fix.extents(H, W)):
                    if wl < W:
                        v[i, :, :hl, wl] = v[i, :, :hl, wl - 1]
                    if hl < H:
                        v[i, :, hl, :min(wl + 1, W)] = v[i, :, hl - 1, :min(wl + 1, W)]
            new[k] = v
        return new
    model.backbone.register_forward_hook(zero_outputs)
    g = sum(_masked_groupnorm(h, fix) for h in heads) + _masked_groupnorm(model.backbone, fix)
    if hasattr(model, "rpn"):
        _own_anchors(model.rpn.anchor_generator, fix); _suppress_padding_scores(model.rpn.head.cls_logits, fix)
    else:
        _own_anchors(model.anchor_generator, fix); _suppress_padding_scores(model.head.classification_head.cls_logits, fix)
    fix.SELF = ("conv", "norm", "own") if g else ("conv", "own")

def install_fix_hf(model, fix):
    enc = model.model.backbone
    _mask_conv_inputs(enc, fix)
    orig = enc.forward
    def fwd(pixel_values, pixel_mask):
        out = orig(pixel_values, pixel_mask)
        if not (fix.exactmask and fix.active()):
            return out
        # the transformer's padding mask with exact extents (the library shrinks it with nearest-neighbour rounding)
        return [(fm, fix.mask(fm)[:, 0].bool()) for fm, _ in out]
    enc.forward = fwd
    fix.SELF = ("conv", "exactmask")

## 4. Detectors

In [ ]:
ALIASES = {"motorbike": "motorcycle", "aeroplane": "airplane", "sofa": "couch", "tvmonitor": "tv", "tv monitor": "tv",
           "diningtable": "dining table", "pottedplant": "potted plant", "hair dryer": "hair drier",
           "sports ball": "sports ball", "cellphone": "cell phone", "mobile phone": "cell phone"}
def _norm(s):
    s = " ".join(str(s).lower().replace("_", " ").replace("-", " ").split())
    return ALIASES.get(s, s)
def names_to_cat(names, label=""):
    m, miss = {}, []
    for k, v in names.items():
        c = NAME2CAT.get(_norm(v))
        if c is not None:
            m[int(k)] = c
        elif _norm(v) not in ("n/a", "__background__", "background", "no object"):
            miss.append(v)
    if miss:
        print(f"  unmapped class names in {label}: {miss}")
    return m

def load_hf_model(cls, repo, revision=None):
    try:
        return cls.from_pretrained(repo, revision=revision)
    except Exception as ex:
        if "alidation" not in f"{type(ex).__name__} {ex}":
            raise
        from huggingface_hub import hf_hub_download
        raw = json.load(open(hf_hub_download(repo, "config.json", revision=revision)))
        defaults = {f.name: f.default for f in dataclasses.fields(cls.config_class) if f.default is not dataclasses.MISSING}
        dropped = [k for k, v in raw.items() if v is None and defaults.get(k) is not None]
        print(f"  (config fix for {repo}: replaced null fields {dropped} with library defaults)")
        raw = {k: v for k, v in raw.items() if k not in dropped}
        return cls.from_pretrained(repo, revision=revision, config=cls.config_class.from_dict(raw))

def roundup(x, d):
    return int(math.ceil(x / d) * d)

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

def _pack(boxes, scores, labels, lab2cat):
    lab = labels.cpu().numpy()
    keep = np.array([l in lab2cat for l in lab], bool)
    return {"boxes": boxes.float().cpu().numpy()[keep], "scores": scores.float().cpu().numpy()[keep],
            "labels": np.array([lab2cat[l] for l in lab[keep]], dtype=int)}

class MicroBatch:
    """If a batch does not fit in GPU memory, split it into smaller pieces that are all padded to the canvas the
    full batch would have had. Images in these models never interact except through padding, so this gives the
    same result as the full batch (verified for DETR by the canvas self-test, and for every Part-B model by B0)."""
    micro = None   # largest piece size known to fit; None = no limit found yet

    def batch_canvas(self, imgs):
        sizes = [self.input_size(im) for im in imgs]
        return (roundup(max(h for h, _ in sizes), self.div), roundup(max(w for _, w in sizes), self.div))

    def run(self, imgs, canvas=None):
        if self.fixed_size:
            return self._run(imgs, canvas)
        if self.micro is not None and len(imgs) > self.micro:
            return self._split(imgs, canvas)
        try:
            return self._run(imgs, canvas)
        except torch.cuda.OutOfMemoryError:
            if len(imgs) == 1:
                raise
        # We are now OUTSIDE the except block, so the failed attempt (and the GPU memory it holds) is released
        # before retrying. Retrying inside the except block keeps that memory alive.
        free_mem()
        self.micro = max(1, len(imgs) // 2)
        print(f"  [{self.name}] batch of {len(imgs)} does not fit in GPU memory → running pieces of {self.micro} "
              f"on the full batch's canvas (equivalent result)")
        return self._split(imgs, canvas)

    def _split(self, imgs, canvas):
        canvas = canvas if canvas is not None else self.batch_canvas(imgs)
        step = self.micro                      # fixed for this call, even if a piece lowers self.micro further
        R, P = [], []
        for k in range(0, len(imgs), step):
            r, p = self.run(imgs[k:k + step], canvas)
            R += r; P += p
        assert len(R) == len(imgs)
        return R, P

class HFDet(MicroBatch):
    kind, div = "hf", 1
    def __init__(self, name, repo, cls_name, revision=None, fixed_size=False, fixable=True, proc_cls="AutoImageProcessor"):
        self.name, self.fixed_size = name, fixed_size
        self.proc = getattr(transformers, proc_cls).from_pretrained(repo, revision=revision)
        self.model = load_hf_model(getattr(transformers, cls_name), repo, revision).to(DEVICE).eval()
        self.lab2cat = names_to_cat(self.model.config.id2label, name)
        self.accepts_mask = "pixel_mask" in inspect.signature(self.model.forward).parameters
        self.fix = Fix(); self.fix.SELF = None
        if fixable and not fixed_size:
            install_fix_hf(self.model, self.fix)

    # --- hooks that model-specific subclasses override ---
    def _improc(self):
        return self.proc
    def _extra(self, n):
        return {}
    def _post(self, out, imgs, kw):
        sizes = torch.tensor([[im.size[1], im.size[0]] for im in imgs])
        res = self.proc.post_process_object_detection(out, threshold=0.0, target_sizes=sizes)
        return [_pack(r["boxes"], r["scores"], r["labels"], self.lab2cat) for r in res]

    def _single(self, img):
        enc = self._improc()(images=[img], return_tensors="pt")
        pv = enc["pixel_values"][0]
        m = enc["pixel_mask"][0] if "pixel_mask" in enc else torch.ones(pv.shape[-2:], dtype=torch.long)
        return pv, m

    def input_size(self, img):
        return tuple(self._single(img)[0].shape[-2:])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        if canvas is None:
            enc = self._improc()(images=imgs, return_tensors="pt")
            pv = enc["pixel_values"]
            mask = enc["pixel_mask"] if "pixel_mask" in enc else None
        else:
            H, W = canvas; pvs, ms = [], []
            for im in imgs:
                p, m = self._single(im); h, w = p.shape[-2:]
                assert h <= H and w <= W, f"canvas {canvas} smaller than image {(h, w)}"
                pvs.append(F.pad(p, (0, W - w, 0, H - h))); ms.append(F.pad(m, (0, W - w, 0, H - h)))
            pv, mask = torch.stack(pvs), torch.stack(ms)
        kw = {"pixel_values": pv.to(DEVICE)}
        if mask is not None and self.accepts_mask:
            kw["pixel_mask"] = mask.to(DEVICE)
            if self.fix.any_on():
                self.fix.begin([(int(mask[i, :, 0].sum()), int(mask[i, 0, :].sum())) for i in range(len(imgs))], pv.shape[-2:])
        kw.update(self._extra(len(imgs)))
        try:
            out = self.model(**kw)
        finally:
            self.fix.end()
        pad = (1 - mask.float().mean(dim=(1, 2))).tolist() if mask is not None else [0.0] * len(imgs)
        return self._post(out, imgs, kw), pad

COCO_PROMPT = " . ".join(c["name"] for c in GT.loadCats(GT.getCatIds())) + " ."

class GroundingDINODet(HFDet):
    """Open-vocabulary detector prompted with all 80 COCO class names."""
    def __init__(self, name, repo):
        super().__init__(name, repo, "GroundingDinoForObjectDetection", fixable=False, proc_cls="AutoProcessor")
        self.lab2cat = {}
        n_tok = len(self.proc.tokenizer(COCO_PROMPT)["input_ids"])
        print(f"  prompt: {n_tok} tokens (model limit {getattr(self.model.config, 'max_text_len', '?')})")
    def _improc(self):
        return self.proc.image_processor
    def _extra(self, n):
        t = self.proc.tokenizer([COCO_PROMPT] * n, padding="longest", return_tensors="pt")
        return {k: v.to(DEVICE) for k, v in t.items()}
    def _post(self, out, imgs, kw):
        sizes = [(im.size[1], im.size[0]) for im in imgs]
        res = self.proc.post_process_grounded_object_detection(out, input_ids=kw["input_ids"], threshold=0.05,
                                                              text_threshold=0.25, target_sizes=sizes)
        packed = []
        for r in res:
            names = r.get("text_labels", r.get("labels"))
            cats = [NAME2CAT.get(_norm(n)) for n in names]
            keep = np.array([c is not None for c in cats], bool)
            packed.append({"boxes": r["boxes"].float().cpu().numpy()[keep], "scores": r["scores"].float().cpu().numpy()[keep],
                           "labels": np.array([c for c in cats if c is not None], dtype=int)})
        return packed

class Mask2FormerDet(HFDet):
    """Instance segmentation; boxes are taken from the predicted masks."""
    def __init__(self, name, repo):
        super().__init__(name, repo, "Mask2FormerForUniversalSegmentation", fixable=False)
    def _post(self, out, imgs, kw):
        sizes = [(im.size[1], im.size[0]) for im in imgs]
        res = self.proc.post_process_instance_segmentation(out, threshold=0.1, target_sizes=sizes, return_binary_maps=True)
        packed = []
        for r in res:
            seg = r["segmentation"]; info = r["segments_info"]
            boxes, scores, labels = [], [], []
            if seg is not None and len(info):
                segs = seg.bool().cpu().numpy() if seg.dim() == 3 else None
                for k, s in enumerate(info):
                    m = segs[k] if segs is not None else (seg == s["id"]).cpu().numpy()
                    c = self.lab2cat.get(int(s["label_id"]))
                    if c is None or not m.any():
                        continue
                    ys, xs = np.where(m)
                    boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]); scores.append(s["score"]); labels.append(c)
            packed.append({"boxes": np.array(boxes, np.float32).reshape(-1, 4), "scores": np.array(scores, np.float32),
                           "labels": np.array(labels, int)})
        return packed

class TVDet(MicroBatch):
    kind, div, fixed_size = "tv", 32, False
    with_masks = False
    def __init__(self, name, builder, weights):
        self.name = name
        self.model = builder(weights=weights).to(DEVICE).eval()
        self.tf = weights.transforms()
        self.lab2cat = names_to_cat(dict(enumerate(weights.meta["categories"])), name)
        self._setup()

    def _setup(self):
        self._canvas = None
        self.fix = Fix()
        orig = self.model.transform.batch_images
        def batch_images(images, size_divisible=32):
            b = orig(images, size_divisible)
            H, W = self._canvas if self._canvas is not None else tuple(b.shape[-2:])
            assert b.shape[-2] <= H and b.shape[-1] <= W, f"canvas {(H, W)} smaller than batch {tuple(b.shape[-2:])}"
            if (H, W) == tuple(b.shape[-2:]):
                return b
            return F.pad(b, (0, W - b.shape[-1], 0, H - b.shape[-2]))
        self.model.transform.batch_images = batch_images
        install_fix_tv(self.model, self.fix)

    def _pack_masks(self, o):
        lab = o["labels"].cpu().numpy(); sc = o["scores"].float().cpu().numpy()
        keep = np.array([l in self.lab2cat for l in lab], bool) & (sc >= 0.05)
        ms = (o["masks"][:, 0] > 0.5).cpu().numpy()[keep]
        rles = []
        for m in ms:
            r = mask_utils.encode(np.asfortranarray(m.astype(np.uint8))); r["counts"] = r["counts"].decode("ascii"); rles.append(r)
        return {"boxes": o["boxes"].float().cpu().numpy()[keep], "scores": sc[keep],
                "labels": np.array([self.lab2cat[l] for l in lab[keep]], dtype=int), "masks": rles}

    @torch.inference_mode()
    def input_size(self, img):
        il, _ = self.model.transform([img if torch.is_tensor(img) else self.tf(img).to(DEVICE)])
        return tuple(il.image_sizes[0])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        self._canvas = canvas
        try:
            x = [im if torch.is_tensor(im) else self.tf(im).to(DEVICE) for im in imgs]
            il, _ = self.model.transform(x)
            H, W = il.tensors.shape[-2:]
            pad = [1 - (h * w) / (H * W) for h, w in il.image_sizes]
            if self.fix.any_on():
                d = self.model.transform.size_divisible
                sizes = [tuple(s) for s in il.image_sizes]
                self.fix.begin([(roundup(h, d), roundup(w, d)) for h, w in sizes] if self.fix.own else sizes, (H, W))
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP and DEVICE == "cuda"):
                out = self.model(x)
        finally:
            self._canvas = None; self.fix.end()
        if self.with_masks:
            return [self._pack_masks(o) for o in out], pad
        return [_pack(o["boxes"], o["scores"], o["labels"], self.lab2cat) for o in out], pad

def _tv(builder_name, weights_name, name):
    import torchvision.models.detection as D
    return lambda: TVDet(name, getattr(D, builder_name), getattr(D, weights_name).DEFAULT)

LOADERS = {
    # models from Exps 1-3 (used here for the mask and speed parts)
    "detr_r50": lambda: HFDet("detr_r50", "facebook/detr-resnet-50", "DetrForObjectDetection", revision="no_timm"),
    "frcnn_v2": _tv("fasterrcnn_resnet50_fpn_v2", "FasterRCNN_ResNet50_FPN_V2_Weights", "frcnn_v2"),
    "maskrcnn_v2": _tv("maskrcnn_resnet50_fpn_v2", "MaskRCNN_ResNet50_FPN_V2_Weights", "maskrcnn_v2"),
    "retinanet_v2": _tv("retinanet_resnet50_fpn_v2", "RetinaNet_ResNet50_FPN_V2_Weights", "retinanet_v2"),
    # new in Exp 4
    "conditional_detr": lambda: HFDet("conditional_detr", "microsoft/conditional-detr-resnet-50", "ConditionalDetrForObjectDetection"),
    "dab_detr": lambda: HFDet("dab_detr", "IDEA-Research/dab-detr-resnet-50", "DabDetrForObjectDetection", fixable=False),
    "yolos_small": lambda: HFDet("yolos_small", "hustvl/yolos-small", "YolosForObjectDetection", fixable=False),
    "grounding_dino_tiny": lambda: GroundingDINODet("grounding_dino_tiny", "IDEA-Research/grounding-dino-tiny"),
    "mask2former_swin_s": lambda: Mask2FormerDet("mask2former_swin_s", "facebook/mask2former-swin-small-coco-instance"),
    "rtdetr_v2_r50": lambda: HFDet("rtdetr_v2_r50", "PekingU/rtdetr_v2_r50vd", "RTDetrV2ForObjectDetection", fixed_size=True),
}
# Published full COCO val2017 box AP; our 500-image batch-1 AP should be within about ±2.
REF_AP = {"detr_r50": 42.0, "frcnn_v2": 46.7, "maskrcnn_v2": 47.4, "retinanet_v2": 41.5,
          "conditional_detr": 41.0, "dab_detr": 42.2, "yolos_small": 36.1, "grounding_dino_tiny": 48.4,
          "mask2former_swin_s": None, "rtdetr_v2_r50": 53.4}

def load(name):
    free_mem()
    t0 = time.time()
    try:
        det = LOADERS[name]()
        det.micro = CFG["micro_batch"].get(name)
    except Exception as ex:
        print(f"✗ {name} failed to load: {type(ex).__name__}: {str(ex)[:300]}  — skipping this model")
        return None
    note = f"; runs in pieces of {det.micro} (padded to the full batch's canvas)" if det.micro else ""
    print(f"\n== {name} loaded in {time.time() - t0:.0f}s; mapped classes {len(set(det.lab2cat.values()))}/80{note}")
    return det

def free_mem():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 5. Metrics and helpers

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(p, q, thr, iou_min=0.5):
    ia = np.where(p["scores"] >= thr)[0]; ib = np.where(q["scores"] >= thr)[0]
    ia = ia[np.argsort(-p["scores"][ia])]
    M = iou_matrix(p["boxes"][ia], q["boxes"][ib]); used = set(); pairs = []
    for r, i in enumerate(ia):
        best, bc = -1.0, None
        for c, j in enumerate(ib):
            if c not in used and p["labels"][i] == q["labels"][j] and M[r, c] > best:
                best, bc = M[r, c], c
        if bc is not None and best >= iou_min:
            used.add(bc); pairs.append((i, ib[bc], best))
    return pairs, ia, ib

def compare(ref, new):
    """Image-level flip + detection-level change counts + max score change, ref vs new."""
    pairs, ia, ib = greedy_match(ref, new, CFG["flip_score_thr"])
    stable = sum(iou >= CFG["flip_iou"] for _, _, iou in pairs)
    moved = len(pairs) - stable
    lost, gained = len(ia) - len(pairs), len(ib) - len(pairs)
    allp, _, _ = greedy_match(ref, new, 0.05)
    dscore = max([abs(ref["scores"][i] - new["scores"][j]) for i, j, _ in allp], default=0.0)
    return {"flip": (moved + lost + gained) > 0, "stable": stable, "moved": moved, "lost": lost, "gained": gained,
            "max_score_delta": dscore}

def det_survival(ref, new, W, H):
    """For each confident ref detection: did it survive unchanged? plus its distance to the right/bottom edge."""
    pairs, ia, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ok = {i for i, _, iou in pairs if iou >= CFG["flip_iou"]}
    rows = []
    for i in ia:
        x1, y1, x2, y2 = ref["boxes"][i]
        rows.append({"changed": i not in ok, "d_right": (W - x2) / W, "d_bottom": (H - y2) / H,
                     "rel_area": (x2 - x1) * (y2 - y1) / (W * H), "score": ref["scores"][i]})
    return rows

def coco_ap(preds, idx=None):
    idx = list(range(N)) if idx is None else idx
    dets = []
    for k in idx:
        r = preds[k]; order = np.argsort(-r["scores"])[:100]
        for t in order:
            x1, y1, x2, y2 = r["boxes"][t]
            dets.append({"image_id": IDS[k], "category_id": int(r["labels"][t]),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(r["scores"][t])})
    if not dets:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "bbox"); E.params.imgIds = [IDS[k] for k in idx]
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

def run_batches(det, batches, canvas_fn=None, targets=None):
    """batches: list of index lists. canvas_fn(batch)->(H,W) or None. targets: optional list of index lists
    saying which members of each batch to record (others are companions). Returns preds, pads, seconds."""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([IMGS[i] for i in b], canvas=canvas_fn(b) if canvas_fn else None)
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

In [ ]:
B = CFG["batch_size"]
def chunks(order, k=B):
    order = list(order)
    return [order[i:i + k] for i in range(0, len(order), k)]

def run_on(det, batches, source, targets=None):
    """like run_batches, but images come from `source` (PIL images or GPU tensors)"""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([source[i] for i in b])
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

def warmup(det, source=None):
    source = IMGS if source is None else source
    for _ in range(2):
        det.run(source[:B]); det.run(source[:1])
    sync()

def with_thr(model):
    CFG["flip_score_thr"] = CFG["flip_thr_per_model"].get(model, 0.5)

def mask_compare(ref, new):
    c = compare(ref, new)
    pairs, _, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ious = [float(mask_utils.iou([new["masks"][j]], [ref["masks"][i]], [0])[0][0]) for i, j, _ in pairs]
    return {"flip": c["flip"], "mask_flip": bool(c["flip"] or any(v < CFG["flip_iou"] for v in ious)),
            "min_mask_iou": min(ious, default=1.0), "max_score_delta": c["max_score_delta"]}

def coco_segm_ap(preds):
    dets = []
    for k, img_id in enumerate(IDS):
        r = preds[k]
        for t in np.argsort(-r["scores"])[:100]:
            dets.append({"image_id": img_id, "category_id": int(r["labels"][t]), "segmentation": r["masks"][t],
                         "score": float(r["scores"][t])})
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "segm"); E.params.imgIds = IDS
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

bs1 = [[i] for i in range(N)]
rand0 = chunks(np.random.default_rng(0).permutation(N))
aspect = np.array([im.size[0] / im.size[1] for im in IMGS]); area = np.array([im.size[0] * im.size[1] for im in IMGS])
sorted_b = chunks(np.lexsort((area, aspect)))
WIDEST, TALLEST = int(np.argmax(aspect)), int(np.argmin(aspect))
mp = chunks(range(N), B - 2); maxpad = [c + [WIDEST, TALLEST] for c in mp]

## 6. Part A — newer detectors

In [ ]:
if RUN_PARTS["A"]:
    AR = read_csv(f"{OUT}/A_runs.csv").to_dict("records") if os.path.exists(f"{OUT}/A_runs.csv") else []
    AI = read_csv(f"{OUT}/A_per_image.csv").to_dict("records") if os.path.exists(f"{OUT}/A_per_image.csv") else []
    done = {r["model"] for r in AR}
    for mname in CFG["models_A"]:
        if mname in done:
            print(f"== {mname}: already done, skipping"); continue
        det = None; free_mem()
        det = load(mname)
        if det is None:
            continue
        warmup(det)
        plan = [("none", "bs1", bs1, None), ("none", "bs8_rand", rand0, None), ("none", "bs8_sorted", sorted_b, None),
                ("none", "bs8_maxpad", maxpad, mp)]
        if det.fix.SELF:
            plan += [("self", "bs8_rand", rand0, None), ("self", "bs8_maxpad", maxpad, mp)]
        preds = {}
        for vname, bname, batches, targets in plan:
            det.fix.set(det.fix.SELF if vname == "self" else ())
            P, PD, sec = run_on(det, batches, IMGS, targets)
            preds[(vname, bname)] = P
            ap = coco_ap(P)
            ref = f"  (published ≈ {REF_AP.get(mname)})" if (vname, bname) == ("none", "bs1") else ""
            AR.append({"model": mname, "variant": vname, "batching": bname, "AP": ap, "img_per_s": N / sec, "mean_pad": float(np.mean(PD))})
            print(f"  {vname:5s} {bname:11s} AP={ap:6.2f}  pad={np.mean(PD):.3f}  {N / sec:5.1f} img/s{ref}")
        det.fix.set(())
        with_thr(mname)
        for (vname, bname), P in preds.items():
            if (vname, bname) != ("none", "bs1"):
                for i in range(N):
                    AI.append({"model": mname, "variant": vname, "batching": bname, "pad_frac": None,
                               **compare(preds[("none", "bs1")][i], P[i])})
        with_thr(None)
        det = None; preds = None; free_mem()
        pd.DataFrame(AR).to_csv(f"{OUT}/A_runs.csv", index=False); pd.DataFrame(AI).to_csv(f"{OUT}/A_per_image.csv", index=False)

In [ ]:
AR = read_csv(f"{OUT}/A_runs.csv"); AI = read_csv(f"{OUT}/A_per_image.csv")
fa = AI.groupby(["model", "variant", "batching"]).agg(flip=("flip", "mean"), p95_dscore=(
    "max_score_delta", lambda v: np.percentile(v, 95))).reset_index()
ba = AR[(AR.variant == "none") & (AR.batching == "bs1")].set_index("model")["AP"]
AR["dAP"] = AR.apply(lambda r: r.AP - ba[r.model], axis=1)
TA = AR.merge(fa, on=["model", "variant", "batching"], how="left")
display(TA[["model", "variant", "batching", "AP", "dAP", "flip", "p95_dscore", "mean_pad"]].style.format(
    {"AP": "{:.2f}", "dAP": "{:+.2f}", "flip": "{:.1%}", "p95_dscore": "{:.3f}", "mean_pad": "{:.2f}"}, na_rep="—"))
fig, ax = plt.subplots(figsize=(11, 4))
ms = list(dict.fromkeys(AR.model)); w = 0.2
for k, b in enumerate(["bs8_sorted", "bs8_rand", "bs8_maxpad"]):
    y = fa[(fa.variant == "none") & (fa.batching == b)].set_index("model").reindex(ms)["flip"] * 100
    ax.bar(np.arange(len(ms)) + (k - 1) * w, y, w, label=f"normal {b}")
y = fa[(fa.variant == "self") & (fa.batching == "bs8_rand")].set_index("model").reindex(ms)["flip"] * 100
ax.bar(np.arange(len(ms)) + 2 * w, y, w, label="self bs8_rand", color="k")
ax.set_xticks(range(len(ms))); ax.set_xticklabels(ms, rotation=15); ax.set_ylabel("% images changed vs batch 1")
ax.legend(fontsize=8); ax.set_title("Part A — newer detectors")
plt.tight_layout(); plt.savefig(f"{OUT}/A_breadth.png", dpi=180); plt.show()

## 7. Part B — instance masks (Mask R-CNN)

In [ ]:
if RUN_PARTS["B"] and not os.path.exists(f"{OUT}/B_runs.csv"):
    det = None; free_mem()
    det = load(CFG["model_B"]); det.with_masks = True
    warmup(det)
    plan = [("none", "bs1", bs1, None), ("none", "bs8_rand", rand0, None),
            ("self", "bs8_rand", rand0, None), ("self", "bs8_maxpad", maxpad, mp)]
    BR, BI, preds = [], [], {}
    for vname, bname, batches, targets in plan:
        det.fix.set(det.fix.SELF if vname == "self" else ())
        P, PD, sec = run_on(det, batches, IMGS, targets)
        preds[(vname, bname)] = P
        box_ap, segm_ap = coco_ap(P), coco_segm_ap(P)
        BR.append({"variant": vname, "batching": bname, "box_AP": box_ap, "mask_AP": segm_ap})
        print(f"  {vname:5s} {bname:11s} box AP={box_ap:6.2f}  mask AP={segm_ap:6.2f}")
    det.fix.set(())
    for (vname, bname), P in preds.items():
        if (vname, bname) != ("none", "bs1"):
            for i in range(N):
                BI.append({"variant": vname, "batching": bname, **mask_compare(preds[("none", "bs1")][i], P[i])})
    det = None; preds = None; free_mem()
    pd.DataFrame(BR).to_csv(f"{OUT}/B_runs.csv", index=False); pd.DataFrame(BI).to_csv(f"{OUT}/B_per_image.csv", index=False)

BR = read_csv(f"{OUT}/B_runs.csv"); BI = read_csv(f"{OUT}/B_per_image.csv")
b0 = BR[(BR.variant == "none") & (BR.batching == "bs1")].iloc[0]
BR["d_box_AP"] = BR.box_AP - b0.box_AP; BR["d_mask_AP"] = BR.mask_AP - b0.mask_AP
fb = BI.groupby(["variant", "batching"]).agg(box_flip=("flip", "mean"), mask_flip=("mask_flip", "mean"),
                                             worst_mask_iou=("min_mask_iou", "min")).reset_index()
TB = BR.merge(fb, on=["variant", "batching"], how="left")
display(TB.style.format({"box_AP": "{:.2f}", "mask_AP": "{:.2f}", "d_box_AP": "{:+.3f}", "d_mask_AP": "{:+.3f}",
                         "box_flip": "{:.1%}", "mask_flip": "{:.1%}", "worst_mask_iou": "{:.3f}"}, na_rep="—"))

## 8. Part C — speed where batching pays (half precision, short side 512)

In [ ]:
if RUN_PARTS["C"]:
    CR = read_csv(f"{OUT}/C_runs.csv").to_dict("records") if os.path.exists(f"{OUT}/C_runs.csv") else []
    CI = read_csv(f"{OUT}/C_per_image.csv").to_dict("records") if os.path.exists(f"{OUT}/C_per_image.csv") else []
    done = {r["model"] for r in CR}
    AMP = True
    for mname in CFG["models_C"]:
        if mname in done:
            print(f"== {mname}: already done, skipping"); continue
        det = None; free_mem()
        det = load(mname)
        if det is None:
            continue
        det.model.transform.min_size = (CFG["speed_min_size"],); det.model.transform.max_size = CFG["speed_max_size"]
        with torch.inference_mode():
            XS = [det.tf(im).to(DEVICE) for im in IMGS]         # decode + convert once; timing below is model-only
        d = det.model.transform.size_divisible
        own = [tuple(roundup(s, d) for s in det.input_size(x)) for x in XS]
        groups = {}
        for i, c in enumerate(own):
            groups.setdefault(c, []).append(i)
        exact = [b for g in groups.values() for b in chunks(g)]
        print(f"  {len(groups)} distinct canvases; exact-shape bucketing makes {len(exact)} batches "
              f"(average {N / len(exact):.1f} images per batch, vs {B} when batches are full)")
        warmup(det, XS)
        plan = [("none", "bs1", bs1), ("none", "bs8_rand", rand0), ("none", "bs8_sorted", sorted_b),
                ("none", "exact_shape_buckets", exact), ("self", "bs8_rand", rand0), ("self", "bs8_sorted", sorted_b)]
        preds = {}
        for vname, bname, batches in plan:
            det.fix.set(det.fix.SELF if vname == "self" else ())
            P, PD, sec = run_on(det, batches, XS)
            preds[(vname, bname)] = P
            CR.append({"model": mname, "variant": vname, "batching": bname, "AP": coco_ap(P), "img_per_s": N / sec})
            print(f"  {vname:5s} {bname:20s} AP={CR[-1]['AP']:6.2f}  {N / sec:6.1f} img/s")
        det.fix.set(())
        for (vname, bname), P in preds.items():
            if (vname, bname) != ("none", "bs1"):
                for i in range(N):
                    CI.append({"model": mname, "variant": vname, "batching": bname, **compare(preds[("none", "bs1")][i], P[i])})
        det = None; preds = None; XS = None; free_mem()
        pd.DataFrame(CR).to_csv(f"{OUT}/C_runs.csv", index=False); pd.DataFrame(CI).to_csv(f"{OUT}/C_per_image.csv", index=False)
    AMP = False

CR = read_csv(f"{OUT}/C_runs.csv"); CI = read_csv(f"{OUT}/C_per_image.csv")
fc = CI.groupby(["model", "variant", "batching"]).agg(flip=("flip", "mean")).reset_index()
c1 = CR[(CR.variant == "none") & (CR.batching == "bs1")].set_index("model")
CR["speed_vs_bs1"] = CR.apply(lambda r: r.img_per_s / c1.img_per_s[r.model], axis=1)
CR["dAP"] = CR.apply(lambda r: r.AP - c1.AP[r.model], axis=1)
TC = CR.merge(fc, on=["model", "variant", "batching"], how="left")
display(TC.style.format({"AP": "{:.2f}", "dAP": "{:+.3f}", "img_per_s": "{:.1f}", "speed_vs_bs1": "{:.2f}×", "flip": "{:.2%}"}, na_rep="—"))
fig, ax = plt.subplots(figsize=(9, 4))
for m in TC.model.unique():
    d = TC[TC.model == m]
    ax.scatter(d.speed_vs_bs1, d.flip.fillna(0) * 100, s=60, label=m)
    for _, r in d.iterrows():
        ax.annotate(f"{r.variant} {r.batching}", (r.speed_vs_bs1, (0 if pd.isna(r.flip) else r.flip) * 100), fontsize=7)
ax.set_xlabel("throughput vs batch 1 (half precision, short side 512)"); ax.set_ylabel("% images changed vs batch 1")
ax.set_title("Part C — speed vs exactness"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/C_speed.png", dpi=180); plt.show()

## 9. Claims check

In [ ]:
K = CFG["claims"]; v = {}
new_pad = [m for m in AR.model.unique() if m not in CFG["control_models"]]
fr = fa[(fa.variant == "none") & (fa.batching == "bs8_rand")].set_index("model")["flip"]
n_hi = int((fr.reindex(new_pad) >= K["G1_flip"]).sum())
ctrl = [m for m in CFG["control_models"] if m in fr.index]
ctrl_ok = all(fr[m] < K["G1_ctrl"] for m in ctrl) if ctrl else False
v["G1_breadth"] = f"{'✓' if n_hi >= K['G1_models'] and ctrl_ok else '✗'}  {n_hi}/{len(new_pad)} new models ≥{K['G1_flip']:.0%}; control {[f'{fr[m]:.2%}' for m in ctrl]}"
s = TA[(TA.variant == "self")]
if len(s):
    ok = (s.flip.max() <= K["G2_flip"]) and (s.dAP.abs().max() <= K["G2_dap"])
    v["G2_fix_transfers"] = f"{'✓' if ok else '✗'}  worst flip {s.flip.max():.2%}, worst |ΔAP| {s.dAP.abs().max():.3f} ({', '.join(s.model.unique())})"
else:
    v["G2_fix_transfers"] = "✗  no model with the fix ran"
nb = TB[(TB.variant == "none") & (TB.batching == "bs8_rand")].iloc[0]; sb = TB[TB.variant == "self"]
ok = nb.mask_flip >= K["G3_normal"] and sb.mask_flip.max() <= K["G3_self"] and sb.d_mask_AP.abs().max() <= K["G3_dap"]
v["G3_masks"] = f"{'✓' if ok else '✗'}  normal {nb.mask_flip:.1%} of images' masks change; self {sb.mask_flip.max():.2%}, |Δ mask AP| {sb.d_mask_AP.abs().max():.3f}"
g4 = []
for m in TC.model.unique():
    d = TC[TC.model == m].set_index(["variant", "batching"])
    a = d.loc[("none", "bs8_sorted"), "speed_vs_bs1"] >= K["G4_batching"]
    b = d.loc[("self", "bs8_sorted"), "img_per_s"] / d.loc[("none", "bs8_sorted"), "img_per_s"] >= K["G4_self"]
    c = max(d.loc[("self", "bs8_rand"), "flip"], d.loc[("self", "bs8_sorted"), "flip"]) <= K["G4_exact"]
    g4.append(a and b and c)
    print(f"  {m}: sorted bs8 {d.loc[('none', 'bs8_sorted'), 'speed_vs_bs1']:.2f}× bs1 | self sorted "
          f"{d.loc[('self', 'bs8_sorted'), 'img_per_s'] / d.loc[('none', 'bs8_sorted'), 'img_per_s']:.2f}× normal sorted | "
          f"exact buckets {d.loc[('none', 'exact_shape_buckets'), 'speed_vs_bs1']:.2f}× bs1 | self flips "
          f"{max(d.loc[('self', 'bs8_rand'), 'flip'], d.loc[('self', 'bs8_sorted'), 'flip']):.2%}")
v["G4_speed"] = f"{'✓' if all(g4) else '✗'}  {sum(g4)}/{len(g4)} models"
for k, x in v.items():
    print(f"{k:18s} {x}")
with open(f"{OUT}/manifest.json", "w") as f:
    json.dump({"env": ENV, "config": CFG, "image_ids": IDS, "verdicts": v}, f, indent=2, default=str)
print("saved:", sorted(os.listdir(OUT)))

## What to bring back

Upload the finished notebook (or the `exp4` folder). Key outputs: the Part A table and figure, the Part B table, the Part C table and figure, and the claim lines.